In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import glob
import scipy.io
import os 
import anndata 

In [2]:
os.chdir('/lustre/groups/ml01/workspace/hpca')

In [3]:
def create_and_save_adata(csv_file, save_path=None):

    
    if isinstance(csv_file, str):
        if csv_file.endswith('.txt'):
            print(f'{csv_file} ends with txt, using read_table')
            csv = pd.read_table(csv_file, index_col=0)
        elif csv_file.endswith('.csv'):
            print(f'{csv_file} ends with csv, using read_csv')
            csv = pd.read_csv(csv_file, index_col=0)            
    elif isinstance(csv_file, pd.DataFrame):
        csv = csv_file
    else:
        raise ValueError("csv_input must be a file path (str) or a pandas DataFrame")    
    obs = pd.DataFrame()
    obs['sample'] = csv.columns
    var_names = csv.index
    var = pd.DataFrame(index=var_names)
    var = var.astype(str)
    X = csv.T.values
    adata = anndata.AnnData(X, obs=obs, var=var)
    print(adata.obs)
    if save_path:
        adata.write(save_path)
        print(f"AnnData object saved to {save_path}")

    return adata


In [4]:
class DataProcessor:
    def __init__(self, root_directory):
        self.root_directory = root_directory

    def assemble_h5ad(self, barcodes_file, features_file, matrix_file, output_file):
        # Read barcode, features, and matrix files
        barcodes = pd.read_csv(barcodes_file, header=None, index_col=0, names=['barcode'])
        features = pd.read_csv(features_file, sep='\t', header=None)
        features.columns = features.columns.astype(str)
        barcodes.columns = barcodes.columns.astype(str)
        matrix = scipy.io.mmread(matrix_file).T.tocsc()

        # Create AnnData object
        adata = anndata.AnnData(X=matrix, obs=barcodes, var=features)

        # Convert index to strings
        adata.var = adata.var.astype(str)
        adata.obs = adata.obs.astype(str)

        # Write h5ad file
        adata.write_h5ad(output_file)

    def process_directory(self):
        file_counter = 0 
        # Traverse the root directory
        for dataset_dir in os.listdir(self.root_directory):
            dataset_path = os.path.join(self.root_directory, dataset_dir)
            if not os.path.isdir(dataset_path):
                continue

            # Traverse Gene/GeneFull directories
            for gene_type in ['Gene', 'GeneFull']:
                gene_type_path = os.path.join(dataset_path, 'output', gene_type)
                if not os.path.isdir(gene_type_path):
                    continue

                # Traverse filtered/raw directories
                for data_type in ['filtered', 'raw']:
                    data_type_path = os.path.join(gene_type_path, data_type)
                    if not os.path.isdir(data_type_path):
                        continue

                    # Find matrix, barcodes, and features files
                    matrix_files = [f for f in os.listdir(data_type_path) if f.endswith(('matrix.mtx.gz', 'matrix.mtx'))]
                    if not matrix_files:
                        print(f'No matrix file in {data_type_path}, skipping this directory')
                        continue

                    matrix_file = os.path.join(data_type_path, matrix_files[0])
                    barcodes_file = os.path.join(data_type_path, [f for f in os.listdir(data_type_path) if 'barcode' in f or 'barcodes' in f][0])
                    features_file = os.path.join(data_type_path, [f for f in os.listdir(data_type_path) if 'feature' in f or 'genes' in f][0])

                    # Define the output file path with indication of raw/filtered and Gene/GeneFull
                    output_file = os.path.join(self.root_directory, dataset_dir, f'{dataset_dir}_{gene_type}_{data_type}.h5ad')

                    # Call the assemble_h5ad function
                    self.assemble_h5ad(barcodes_file, features_file, matrix_file, output_file)
                    file_counter += 1
                    print(f'Saving file at: {output_file}')

        print(f"Successfully processed {file_counter} datasets.")

---

# datasets to be use: GSE217837, GSE183568, GSE101207, GSE150724, GSE198623, GSE148073
# GSE114297, GSE217837, GSE84133 (needs redownloading) Charite snRNA, Espace, PANC-DB 


---

# PROBLEMATIC
# GSE233476, GSE97655, GSE86473, GSE101207, 
# GSE84133, GSE83139, GSE81608

## GSE97655: no count matrix, some dataframe   ✔️
## GSE233476: weird .narrowPeak file   ✔️
## GSE101207: processed txt files but not sure what to do with .bw files   ✔️
## GSE86473: Not sure if the columns in the CSV are individual samples   ✔️ 
## GSE84133: Unknown problem, data needs to be redownloaded probably   ✔️
## GSE83139: Not a count matrix   ✔️
## GSE81608: Not a count matrix   ✔️

---

# GSE217837


In [8]:
create_and_save_adata("GSE217837/GSE217837_scRNA_count_matrix.csv", "GSE217837/GSE217837_scRNA_count_matrix.h5ad")

GSE217837/GSE217837_scRNA_count_matrix.csv ends with csv, using read_csv


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


                     sample
0      AAACCCAAGGATATGT.1_1
1      AAACCCACACAAATAG.1_1
2      AAACCCACAGACCAGA.1_1
3      AAACCCAGTCGCACGT.1_1
4      AAACCCAGTCTGTAAC.1_1
...                     ...
10727  TTTGGTTGTCTAGTGT.1_3
10728  TTTGGTTTCACTGTCC.1_3
10729  TTTGGTTTCATTCCTA.1_3
10730  TTTGTTGTCAAATAGG.1_3
10731  TTTGTTGTCTACAGGT.1_3

[10732 rows x 1 columns]
AnnData object saved to GSE217837/GSE217837_scRNA_count_matrix.h5ad


AnnData object with n_obs × n_vars = 10732 × 26671
    obs: 'sample'

In [9]:
create_and_save_adata("GSE217837/GSE217837_snRNA_count_matrix.csv", "GSE217837/GSE217837_snRNA_count_matrix.h5ad")

GSE217837/GSE217837_snRNA_count_matrix.csv ends with csv, using read_csv


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


                     sample
0      AAACCCACATGGCACC.1_4
1      AAACCCAGTACAAACA.1_4
2      AAACCCAGTCATCCGG.1_4
3      AAACCCATCTGATTCT.1_4
4      AAACGAAAGACGGAAA.1_4
...                     ...
11013  TTTGGAGAGAGAGGGC.1_6
11014  TTTGGAGCATCCAATG.1_6
11015  TTTGGTTCAATCTCTT.1_6
11016  TTTGGTTGTTATCCAG.1_6
11017  TTTGTTGCAATTGGTC.1_6

[11018 rows x 1 columns]
AnnData object saved to GSE217837/GSE217837_snRNA_count_matrix.h5ad


AnnData object with n_obs × n_vars = 11018 × 26671
    obs: 'sample'

# GSE183568

In [14]:
directory = 'GSE183568'
for file in os.listdir(directory):
    try:
        if file.endswith('.txt'):
            file_path = os.path.join(directory, file)
            print(f'Reading: {file_path}')
            csv = pd.read_table(file_path)
            save_path = os.path.join(directory, file.replace('.txt', '.h5ad'))
            print(f'Saving file to: {save_path}')
            create_and_save_adata(csv, save_path)
    except Exception as e:
        print(f"An error occurred with {file}: {e}")
    

Reading: GSE183568/GSE183568_raw_gene_cell_counts.txt
Saving file to: GSE183568/GSE183568_raw_gene_cell_counts.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


                    sample
0       AAACCTGGTCTGATTG-1
1       AAACCTGGTCTGGAGA-1
2       AAACGGGAGAATTCCC-1
3       AAACGGGCAACGATCT-1
4       AAACGGGCAAGTCATC-1
...                    ...
44948  TTTGTCAGTAGGCATG-14
44949  TTTGTCAGTTCTGGTA-14
44950  TTTGTCATCCTTGGTC-14
44951  TTTGTCATCGATAGAA-14
44952  TTTGTCATCGCACTCT-14

[44953 rows x 1 columns]
AnnData object saved to GSE183568/GSE183568_raw_gene_cell_counts.h5ad
Reading: GSE183568/GSE183568_scRNAseq_metadata.txt
Saving file to: GSE183568/GSE183568_scRNAseq_metadata.h5ad
       sample
0  Library_ID
1         Age
2         Sex
3       Batch
4  Replicates
5   CellTypes


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


An error occurred with GSE183568_scRNAseq_metadata.txt: Can't implicitly convert non-string objects to strings
Reading: GSE183568/GSE183568_FACS_raw_gene_cell_counts.txt
Saving file to: GSE183568/GSE183568_FACS_raw_gene_cell_counts.h5ad
                              sample
0      FACS-Beta_AAACCTGAGATAGTCA-10
1      FACS-Beta_AAACCTGAGTGCAAGC-10
2      FACS-Beta_AAACGGGAGCTCAACT-10
3      FACS-Beta_AAACGGGGTATTACCG-10
4      FACS-Beta_AAAGATGAGTATCGAA-10
...                              ...
10317  FAC-Alpha_TTTCCTCTCTAACGGT-14
10318  FAC-Alpha_TTTGCGCCATTCACTT-14
10319  FAC-Alpha_TTTGCGCGTTACAGAA-14
10320  FAC-Alpha_TTTGTCACAGACGCTC-14
10321  FAC-Alpha_TTTGTCATCTCTTATG-14

[10322 rows x 1 columns]


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object saved to GSE183568/GSE183568_FACS_raw_gene_cell_counts.h5ad
Reading: GSE183568/GSE183568_FACS_scRNAseq_metadata.txt
Saving file to: GSE183568/GSE183568_FACS_scRNAseq_metadata.h5ad
       sample
0   LibraryID
1         Age
2  Replicates
3    CellType
AnnData object saved to GSE183568/GSE183568_FACS_scRNAseq_metadata.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


# GSE101207
## remapped

In [9]:
root_directory = '/ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE101207/'
processor = DataProcessor(root_directory)
processor.process_directory()

/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE101207/GSM2863189/GSM2863189_Gene_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE101207/GSM2863189/GSM2863189_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE101207/GSM2700342/GSM2700342_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE101207/GSM2700342/GSM2700342_GeneFull_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE101207/GSM2700342/GSM2700342_GeneFull_raw.h5ad
Successfully processed 36 datasets.


# GSE150724

In [19]:
import sys
sys.path.append('/lustre/groups/ml01/workspace/hpca_sara.jimenez_shrey.parikh/')
import atlas_pipeline.preprocess as app

Importing preprocess.py


In [21]:
data_directory = '/lustre/groups/ml01/workspace/hpca/GSE150724/'
processor = app.DataProcessor(root_directory=data_directory)
donor_files = {
    'Donor1': {
        'barcodes': 'GSE150724_Donor1_barcodes.tsv',
        'genes': 'GSE150724_Donor1_genes.tsv',
        'matrix': 'GSE150724_Donor1_matrix.mtx',
    },
    'Donor2': {
        'barcodes': 'GSE150724_Donor2_barcodes.tsv',
        'genes': 'GSE150724_Donor2_genes.tsv',
        'matrix': 'GSE150724_Donor2_matrix.mtx',
    },
    'Donor3': {
        'barcodes': 'GSE150724_Donor3_barcodes.tsv',
        'genes': 'GSE150724_Donor3_genes.tsv',
        'matrix': 'GSE150724_Donor3_matrix.mtx',
    },
}

for donor, files in donor_files.items():
    barcodes_file = os.path.join(data_directory, files['barcodes'])
    genes_file = os.path.join(data_directory, files['genes'])
    matrix_file = os.path.join(data_directory, files['matrix'])
    output_file = os.path.join(data_directory, f'{donor}_output.h5ad')
    
    # Assemble and save the h5ad file
    processor.assemble_h5ad(barcodes_file, genes_file, matrix_file, output_file)
    print(f"Processed and saved {output_file}")


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processed and saved /lustre/groups/ml01/workspace/hpca/GSE150724/Donor1_output.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processed and saved /lustre/groups/ml01/workspace/hpca/GSE150724/Donor2_output.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Processed and saved /lustre/groups/ml01/workspace/hpca/GSE150724/Donor3_output.h5ad


# GSE198623 already h5ad

# GSE148073

## part of HPAP

# GSE114297
## remapped

In [10]:
root_directory = '/ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE114297/'
processor = DataProcessor(root_directory)
processor.process_directory()

/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE114297/GSM3138947/GSM3138947_Gene_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE114297/GSM3138947/GSM3138947_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE114297/GSM3138939/GSM3138939_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE114297/GSM3138939/GSM3138939_GeneFull_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE114297/GSM3138939/GSM3138939_GeneFull_raw.h5ad
Successfully processed 48 datasets.


'/ictstr01/groups/ml01/workspace/hpca'

In [7]:
# GSE114297 from GEO
root_directory = '/ictstr01/groups/ml01/workspace/hpca/data/datasets/GSE114297_remapped/GEO'
processor = DataProcessor(root_directory)
processor.process_directory()

Successfully processed 0 datasets.


In [ ]:
# donor_files = {}

# for file in os.listdir('GSE114297'):
#     if not file.endswith('.tsv') and not file.endswith('.mtx'):
#         continue
#     print(file)
#     parts = file.split('_')
#     donor_id = '_'.join(parts[1:3]) 
#     if donor_id not in donor_files:
#         donor_files[donor_id] = {
#             'barcodes': None,
#             'genes': None,
#             'matrix': None
#         }

#     if 'barcodes' in file:
#         donor_files[donor_id]['barcodes'] = file
#     elif 'genes' in file or 'features' in file:  
#         donor_files[donor_id]['genes'] = file
#     elif 'matrix' in file:
#         donor_files[donor_id]['matrix'] = file

# print(donor_files)

# data_directory = '/lustre/groups/ml01/workspace/hpca/GSE114297/'
# processor = app.DataProcessor(root_directory=data_directory)
# for donor, files in donor_files.items():
#     barcodes_file = os.path.join(data_directory, files['barcodes'])
#     genes_file = os.path.join(data_directory, files['genes'])
#     matrix_file = os.path.join(data_directory, files['matrix'])
#     output_file = os.path.join(data_directory, f'{donor}_output.h5ad')
    
#     # Assemble and save the h5ad file
#     processor.assemble_h5ad(barcodes_file, genes_file, matrix_file, output_file)
#     print(f"Processed and saved {output_file}")


# GSE217837

# already has h5ad files

# GSE84133
## remapped

In [14]:
root_directory = '/ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE84133/'
processor = DataProcessor(root_directory)
processor.process_directory()

/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE84133/GSM2230759/GSM2230759_Gene_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE84133/GSM2230759/GSM2230759_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE84133/GSM2230760/GSM2230760_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE84133/GSM2230760/GSM2230760_GeneFull_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/GSE84133/GSM2230760/GSM2230760_GeneFull_raw.h5ad
Successfully processed 24 datasets.


In [13]:
# test = pd.read_csv('GSE84133/GSM2230757_human1_umifm_counts.csv')

# for file in os.listdir('GSE84133'):
#     if file.endswith('csv'):
#         test = pd.read_csv(os.path.join('GSE84133', file))
#         obs = pd.DataFrame(test.iloc[:,:3])
#         var_names = test.columns[3:]
#         var = pd.DataFrame(index=var_names)
#         X = test.iloc[:,3:].values
#         adata = anndata.AnnData(X, obs=obs, var=var)
#         var = var.astype(str)
#         obs = obs.astype(str)
#         save_path = os.path.join('GSE84133', file.replace('.csv', '.h5ad'))
#         adata.write(save_path)


# test = pd.read_csv('GSE84133/GSM2230759_human3_umifm_counts.csv')
# directory = 'GSE84133'
# for file in os.listdir(directory):
#     try:
#         if file.endswith('csv'):
#             file_path = os.path.join(directory, file)
#             save_path = os.path.join(directory, file.replace('.csv', '.h5ad'))
#             csv = pd.read_csv(file_path)
#             var = pd.DataFrame(csv.columns[3:])
#             var = var.astype(str)
#             # print(var)
#             obs = pd.DataFrame(csv.iloc[:, :3])
#             # print(obs)
#             X = csv.iloc[:, 3:].values
#             adata = anndata.AnnData(X, obs=obs, var=var)
#             print(adata)
#             # print(adata.obs)
#             # print(adata.var)
#             print(f'Saving file to {save_path}')
#             # adata.write(save_path)

#     except Exception as e:
#         print(f'Did not process {file} because {e}')            


# Tabula Sapiens
## remapped

In [16]:
root_directory = '/ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/TabulaSapiens/10x/'
processor = DataProcessor(root_directory)
processor.process_directory()

/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/TabulaSapiens/10x/TSP1_endopancreas_2/TSP1_endopancreas_2_Gene_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/TabulaSapiens/10x/TSP1_endopancreas_2/TSP1_endopancreas_2_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


...

Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/TabulaSapiens/10x/TSP1_exopancreas1_2/TSP1_exopancreas1_2_Gene_raw.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/TabulaSapiens/10x/TSP1_exopancreas1_2/TSP1_exopancreas1_2_GeneFull_filtered.h5ad


/home/aih/shrey.parikh/miniconda3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Saving file at: /ictstr01/groups/ml01/workspace/hpca/remapped/HCA_pancreas_remapped_v1/TabulaSapiens/10x/TSP1_exopancreas1_2/TSP1_exopancreas1_2_GeneFull_raw.h5ad
Successfully processed 44 datasets.


# charite snRNA

In [1]:
import pandas as pd

In [4]:
test = pd.read_csv('charite_snRNA/AdultPancreas/exprMatrix.tsv', sep='\t')

KeyboardInterrupt: 

In [ ]:
test